## Exercícios

> Retirados de [learn-python: sqlalchemy_orm-questions](https://aviadr1.github.io/learn-advanced-python/11_db_access/exercise/sqlalchemy_orm-questions.html).

#### Q1.

Baixa e extraia o arquivo compactado com o banco de dados [Chinook database](https://www.sqlitetutorial.net/sqlite-sample-database/). Salve o arquivo `chinook.db` na mesma pasta deste script.
* Link para baixar: http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip

<img width=500 src=https://www.sqlitetutorial.net/wp-content/uploads/2015/11/sqlite-sample-database-color.jpg>


#### Q2.

Leia o código e os comentários das células a seguir para entender como acessamos os modelos ORM de um banco já existente.

In [ ]:
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.orm import Session

engine = create_engine("sqlite+pysqlite:///chinook.db", echo=False)

### extrai as classes da base de dados Chinook
metadata = MetaData()
metadata.reflect(engine)

# O metadata tem informações sobre as tabelas
# que serão usadas para criar os modelos ORM
for table_name, table in metadata.tables.items():
    print(table_name)
    print(table.columns.keys())
    print(table.columns.items())
    print('-'*25)

### configura o objeto Base mapeando os modelos ORM das tabelas
from sqlalchemy.ext.automap import automap_base
Base = automap_base(metadata=metadata)
Base.prepare()

# o objeto Base tem os modelos ORM que podemos usar
# para manipular o banco de dados
print(Base.classes.items())

In [ ]:
# A seguir um exemplo de query na tabela Albums
# usamos o objeto Base para acessar cada modelo ORM.

session = Session(engine)
res = session.scalars(select(Base.classes.albums))
first_album = res.first()
print(first_album.AlbumId, first_album.Title)

#### Q3. 
Com base nos códigos anteriores realize as operações solicitadas nas células a seguir:


In [ ]:
### Imprima os três primeiros registros da tabela tracks
tracks_3 = session.query(Track).limit(3).all()
for t in tracks_3:
    print(f"ID: {t.TrackId} | Nome: {t.Name}")

In [ ]:
### Imprima o nome da faixa e o título do álbum das primeiras 20 faixas na tabela tracks.
# Realizamos o join entre Track e Album
results = session.query(Track.Name, Album.Title).join(Album).limit(20).all()
for track_name, album_title in results:
    print(f"Faixa: {track_name} | Álbum: {album_title}")

In [ ]:
### Imprima as 10 primeiras vendas de faixas da tabela invoice_items
### Para essas 10 primeiras vendas, imprima os nomes das faixas vendidas e a quantidade vendida.
# 10 primeiras vendas brutas
items = session.query(InvoiceItem).limit(10).all()
for i in items:
    print(f"Invoice: {i.InvoiceId} | TrackID: {i.TrackId} | Qtd: {i.Quantity}")

# Nomes das faixas e quantidades para essas 10 vendas
results_items = session.query(Track.Name, InvoiceItem.Quantity).join(Track).limit(10).all()
for name, qty in results_items:
    print(f"Faixa Vendida: {name} | Quantidade: {qty}")


In [ ]:
### Imprima os nomes das 10 faixas mais vendidas e quantas vezes foram vendidas.
top_tracks = session.query(
    Track.Name, 
    func.sum(InvoiceItem.Quantity).label('total')
).join(InvoiceItem).group_by(Track.Name).order_by(desc('total')).limit(10).all()

print("--- TOP 10 FAIXAS MAIS VENDIDAS ---")
for name, total in top_tracks:
    print(f"{name}: {total} vendas")

In [ ]:
### Quem são os 10 artistas que mais venderam?
### dica: você precisa juntar as tabelas invoice_items, tracks, albums e artists
top_artists = session.query(
    Artist.Name, 
    func.sum(InvoiceItem.Quantity).label('total_vendas')
).join(Album, Artist.ArtistId == Album.ArtistId)\
 .join(Track, Album.AlbumId == Track.AlbumId)\
 .join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId)\
 .group_by(Artist.Name)\
 .order_by(desc('total_vendas'))\
 .limit(10).all()

print("\n--- TOP 10 ARTISTAS QUE MAIS VENDERAM ---")
for name, total in top_artists:
    print(f"{name}: {total} unidades vendidas")